<a href="https://colab.research.google.com/github/OdysseusPolymetis/enexdi_prep_2026/blob/main/3_word_vectors_exercices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercice — Créer et interroger des vecteurs de mots

Dans cet exercice, vous allez créer un petit modèle de **vecteurs de mots** à partir d’un ou plusieurs fichiers `.txt`.

L’objectif est de reprendre les étapes vues dans la démonstration :

1. uploader un corpus ;
2. le concaténer si plusieurs fichiers sont fournis ;
3. le lemmatiser avec Stanza ;
4. transformer le texte en liste de phrases de lemmes ;
5. entraîner un modèle Word2Vec ;
6. interroger les mots proches et les analogies ;
7. exporter les résultats pour TensorFlow Projector.

> Pour l’exercice, on part du principe que le corpus est en français.  
> Si vous utilisez une autre langue, modifiez simplement la variable `langue`.


## 1. Installer les bibliothèques nécessaires

In [ ]:
!pip -q install stanza langdetect gensim tqdm

## 2. Importer les bibliothèques

In [ ]:
from google.colab import files
from pathlib import Path
from tqdm.auto import tqdm

import stanza
from langdetect import detect
from gensim.models import Word2Vec
import zipfile

## 3. Uploader un ou plusieurs fichiers `.txt`

Sélectionnez un ou plusieurs fichiers texte depuis votre ordinateur.


In [ ]:
uploaded = files.upload()

txt_files = []

for filename, content in uploaded.items():
    if filename.endswith(".txt"):
        path = Path("/content") / filename
        path.write_bytes(content)
        txt_files.append(path)

print("Fichiers txt uploadés :")
for path in txt_files:
    print("-", path.name)

## 4. Concaténer les fichiers

Si plusieurs fichiers ont été uploadés, on les regroupe dans un seul texte.


In [ ]:
texts = []

for path in txt_files:
    text = path.read_text(encoding="utf-8")
    texts.append(text)

corpus = "\n\n".join(texts)

print("Nombre de caractères dans le corpus :", len(corpus))
print(corpus[:1000])

## 5. Détecter la langue

La détection automatique n’est pas toujours parfaite, mais elle donne une indication.

Pour simplifier l’exercice, on pourra ensuite forcer la langue en français avec `langue = "fr"`.


In [ ]:
langue_detectee = detect(corpus[:5000])
print("Langue détectée :", langue_detectee)

# Pour cet exercice, on force le français.
# Pour l'anglais : "en" ; pour l'espagnol : "es" ; pour l'italien : "it" ; etc.
langue = "fr"

print("Langue utilisée avec Stanza :", langue)

## 6. Charger le modèle Stanza

Stanza va servir à découper le texte en phrases, en mots, et à récupérer les lemmes.


In [ ]:
stanza.download(langue)

nlp = stanza.Pipeline(
    lang=langue,
    processors="tokenize,mwt,pos,lemma",
    use_gpu=True
)

## 7. Découper le corpus en blocs

On évite d’envoyer tout le texte d’un seul coup à Stanza.


In [ ]:
taille_bloc = 20000

blocs = [
    corpus[i:i+taille_bloc]
    for i in range(0, len(corpus), taille_bloc)
]

print("Nombre de blocs :", len(blocs))

## 8. Lemmatiser le texte

On crée une variable `sentences`, qui contient une liste de phrases.

Chaque phrase est elle-même une liste de mots lemmatisés.

Exemple attendu :

```python
[
    ["le", "père", "aimer", "son", "fils"],
    ["le", "roi", "entrer", "dans", "le", "palais"]
]
```


In [ ]:
sentences = []

for bloc in tqdm(blocs):
    doc = nlp(bloc)

    for sent in doc.sentences:
        phrase = [
            word.lemma.lower()
            for word in sent.words
            if word.upos != "PUNCT"
        ]

        if len(phrase) > 1:
            sentences.append(phrase)

print("Nombre de phrases :", len(sentences))
print(sentences[0])

## 9. Observer les phrases lemmatisées

**Question :** que remarquez-vous ?  
Les mots sont-ils tous bien lemmatisés ?


In [ ]:
for phrase in sentences[:10]:
    print(phrase)

## 10. Entraîner un modèle Word2Vec

`Word2Vec` apprend des représentations vectorielles des mots à partir de leurs contextes d’apparition.

Quelques paramètres importants :

- `vector_size` : dimension des vecteurs ;
- `window` : taille du contexte autour de chaque mot ;
- `min_count` : fréquence minimale pour garder un mot ;
- `sg=1` : modèle skip-gram ;
- `epochs` : nombre de passages sur le corpus.


In [ ]:
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=100
)

print("Modèle entraîné.")
print("Taille du vocabulaire :", len(model.wv.index_to_key))

## 11. Observer le vocabulaire du modèle

**Question :** quels types de mots sont présents dans le vocabulaire ?  
Y a-t-il des mots attendus qui manquent ?


In [ ]:
model.wv.index_to_key[:100]

## 12. Chercher les mots les plus proches

Modifiez la variable `mot` pour tester plusieurs mots du corpus.


In [ ]:
mot = "père"

model.wv.most_similar(mot, topn=20)

## 13. Exercice : tester plusieurs mots

Choisissez trois mots importants dans votre corpus et cherchez leurs voisins vectoriels.

Complétez la cellule ci-dessous.


In [ ]:
mots_a_tester = ["père", "femme", "maison"]

for mot in mots_a_tester:
    print("\n---", mot, "---")

    if mot in model.wv:
        print(model.wv.most_similar(mot, topn=10))
    else:
        print("Mot absent du vocabulaire")

## 14. Tester une analogie

Le modèle peut aussi calculer des relations du type :

> roi - homme + femme ≈ reine

Mais les résultats dépendent beaucoup de la taille et de la qualité du corpus.


In [ ]:
model.wv.most_similar(
    positive=["reine", "homme"],
    negative=["roi"],
    topn=10
)

## 15. Exercice : inventer une analogie

Essayez une analogie adaptée à votre corpus.

Exemples :

```python
positive=["femme", "roi"], negative=["homme"]
positive=["paris", "angleterre"], negative=["france"]
```

Attention : les mots doivent être présents dans le vocabulaire.


In [ ]:
model.wv.most_similar(
    positive=["femme", "roi"],
    negative=["homme"],
    topn=10
)

## 16. Comparer deux mots

On peut calculer la similarité entre deux mots.


In [ ]:
mot1 = "père"
mot2 = "mère"

model.wv.similarity(mot1, mot2)

## 20. Questions finales

Répondez en quelques lignes :

1. Quels mots avez-vous testés ?
2. Les voisins vectoriels vous semblent-ils pertinents ?
3. Quels résultats sont surprenants ?
4. Le corpus est-il assez grand pour produire de bons vecteurs ?
5. Que se passe-t-il si vous changez `min_count`, `window` ou `epochs` ?
